# Aimodel: chat with it, teach it, train its writer

Your private AI that learns from you. Everything here stays in your Kaggle account.
**Keep this notebook private**: its saved output holds your notes.

| Step | What it does | Runs when |
|---|---|---|
| 1 Setup | gets the code | always |
| 2 Restore | brings back your saved brain | always |
| 3 Check | a 30-second test that everything works (including training) | always |
| 4 Chat | talk to it (`CHAT = True`) | only while you are editing |
| 5 Look inside | shows what it knows | always |
| 6 Train | trains your transformer writer (`TRAIN = True`, needs a GPU) | when you switch it on |

**Kaggle settings you need:** *Internet* on (to get the code and for web lookups). For step 6 also set
*Accelerator* to a GPU (it asks you to verify your phone number once).

## 1. Setup

In [ ]:
import os, sys, glob, shutil, subprocess

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("aimodel_work")  # saved with the notebook
CODE = "/tmp/aimodel_code"                                                 # not saved
REPO = "https://github.com/iamsabroy-gif/Aimodel.git"
os.makedirs(WORK, exist_ok=True)

if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "-b", "main", REPO, CODE], check=True)
sys.path.insert(0, CODE)

BRAIN = os.path.join(WORK, "brain.json")
WRITER = os.path.join(WORK, "writer.npz")
print("Code ready. Your brain lives at", BRAIN)

## 2. Restore your saved brain

A new Kaggle session starts empty. To continue where you left off, attach your earlier brain
(**Add Input** → a dataset or notebook output holding `brain.json`). If none is attached, you start fresh.

In [ ]:
def restore():
    # a writer you trained (writer.npz) works with or without a saved brain
    writers = glob.glob("/kaggle/input/**/writer.npz", recursive=True)
    if writers and not os.path.exists(WRITER):
        shutil.copy(writers[0], WRITER)
        print("Restored your trained writer:", os.path.basename(os.path.dirname(writers[0])))
    if os.path.exists(BRAIN):
        print("Using the brain already in", WORK)
        return
    found = glob.glob("/kaggle/input/**/brain.json", recursive=True)
    if not found:
        print("No saved brain found, starting fresh (this is normal the first time).")
        return
    src = os.path.dirname(found[0])
    copied = []
    for f in glob.glob(src + "/brain*") + glob.glob(src + "/writer.npz"):
        shutil.copy(f, WORK)
        copied.append(os.path.basename(f))
    print("Restored", ", ".join(copied), "from", src)

restore()


## 3. Check that everything works

Teaches a tiny demo model, takes an exam, then trains and loads a toy writer. It uses a temporary folder,
so your brain is not touched.

In [ ]:
import tempfile, contextlib, io
from aimodel import LearningModel

def check():
    m = LearningModel(seed=0)
    m.add_document("Cats are mammals. Mammals are warm-blooded animals. Animals are living things. "
                   "Rex is a golden retriever. The sky is blue because air scatters blue light.", "demo.txt")
    print("hello ->", m.respond("hello")[0])
    print("Is a cat an animal? ->", m.respond("Is a cat an animal?")[0])
    exam = m.quiz(5)
    print(f"Self-exam: {exam['score']}/{exam['total']}")
    try:
        import torch
    except ImportError:
        print("PyTorch is not installed here, so training is unavailable (using the model still works).")
        return
    from aimodel import train_writer
    with tempfile.TemporaryDirectory() as d:
        m.export_training_data(d)
        with contextlib.redirect_stdout(io.StringIO()):
            train_writer.main(["--data", d, "--out", d + "/w.npz", "--size", "tiny", "--steps", "30", "--batch", "4"])
        m.load_writer(d + "/w.npz")
        device = "a GPU" if torch.cuda.is_available() else "the CPU"
        print(f"Trained and loaded a toy writer ({m.writer.n_params:,} parameters) on {device}.")
    print("Everything works.")

check()

## 4. Chat

Set `CHAT = True` and run the cell. A text box appears under it. Type `/help` for commands, `/quit` to stop.
(This only works while you are editing the notebook, not in a saved run.)

In [ ]:
import os, sys, subprocess
if "BRAIN" not in globals():  # Kaggle starts a fresh session each time: set up again
    print("Fresh session: setting up again. (If you have a saved brain, run step 2 first.)")
    WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.abspath("aimodel_work")
    CODE = "/tmp/aimodel_code"
    os.makedirs(WORK, exist_ok=True)
    if not os.path.isdir(CODE):
        subprocess.run(["git", "clone", "-q", "-b", "main",
                        "https://github.com/iamsabroy-gif/Aimodel.git", CODE], check=True)
    sys.path.insert(0, CODE)
    BRAIN, WRITER = os.path.join(WORK, "brain.json"), os.path.join(WORK, "writer.npz")

CHAT = False

if CHAT:
    from aimodel.cli import main
    main(["--brain", BRAIN])
else:
    print("Set CHAT = True above and run this cell to chat with your model.")

## 5. Look inside, or use it from code

Teach it your own files by attaching them (**Add Input**) and uncommenting the `m.read(...)` line.

In [ ]:
import os, sys, subprocess
if "BRAIN" not in globals():  # Kaggle starts a fresh session each time: set up again
    print("Fresh session: setting up again. (If you have a saved brain, run step 2 first.)")
    WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.abspath("aimodel_work")
    CODE = "/tmp/aimodel_code"
    os.makedirs(WORK, exist_ok=True)
    if not os.path.isdir(CODE):
        subprocess.run(["git", "clone", "-q", "-b", "main",
                        "https://github.com/iamsabroy-gif/Aimodel.git", CODE], check=True)
    sys.path.insert(0, CODE)
    BRAIN, WRITER = os.path.join(WORK, "brain.json"), os.path.join(WORK, "writer.npz")
from aimodel import LearningModel

m = LearningModel.load(BRAIN)
for key, value in m.stats().items():
    print(f"{key}: {value}")

# m.read("/kaggle/input/<your dataset>/notes.txt")          # teach it a file
# reply, confidence = m.respond("What is my dog called?")   # ask it something
# print(reply)
# m.save(BRAIN)                                              # keep what it learned

## 6. Train your transformer writer

The writer words answers in your own tiny transformer, trained **only** on your data. Switch on a GPU
(Settings → Accelerator), set `TRAIN = True` and run. Each answer it drafts is checked against the
evidence, and bad drafts fall back to the template answer.

Run it again later, after you have taught the model more and rated answers with `/good` and `/bad`:
it keeps what it learned (`--resume`) and improves.

In [ ]:
import os, sys, subprocess
if "BRAIN" not in globals():  # Kaggle starts a fresh session each time: set up again
    print("Fresh session: setting up again. (If you have a saved brain, run step 2 first.)")
    WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.path.abspath("aimodel_work")
    CODE = "/tmp/aimodel_code"
    os.makedirs(WORK, exist_ok=True)
    if not os.path.isdir(CODE):
        subprocess.run(["git", "clone", "-q", "-b", "main",
                        "https://github.com/iamsabroy-gif/Aimodel.git", CODE], check=True)
    sys.path.insert(0, CODE)
    BRAIN, WRITER = os.path.join(WORK, "brain.json"), os.path.join(WORK, "writer.npz")
from aimodel import LearningModel

TRAIN = False
SIZE, STEPS = "base", 6000

if TRAIN:
    import torch
    print("GPU available:", torch.cuda.is_available())
    m = LearningModel.load(BRAIN)
    data = os.path.join(WORK, "writer_data")
    info = m.export_training_data(data)
    print(info)
    if info["examples"] < 20:
        raise SystemExit(f"Only {info['examples']} training examples so far: not enough to train on. "
                         "Teach it first: chat (step 4), read some of your notes (step 5), then come back.")
    cmd = [sys.executable, "-m", "aimodel.train_writer", "--data", data, "--out", WRITER,
           "--size", SIZE, "--steps", str(STEPS)]
    if os.path.exists(WRITER):
        cmd += ["--resume", WRITER]
    subprocess.run(cmd, check=True, cwd=CODE)
    m.load_writer(WRITER)
    m.save(BRAIN)   # the brain now remembers its writer
    print("Done. Your model now uses the writer. Download brain.json, brain.neural.npz and brain.writer.npz")
else:
    print("Set TRAIN = True above (and switch on a GPU) to train your writer.")

## Keep it for next time

Kaggle deletes everything when the session ends unless you **Save Version**. After saving, the files in
this notebook's **Output** (`brain.json`, `brain.neural.npz`, and `brain.writer.npz` or `writer.npz` once
trained) can be downloaded, or turned into a private dataset to attach as an input next time (step 2).
You can also use the same files on your phone.